# Predicting Student Test Scores 
## Score: 8.70680

In [1]:
import time
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression


In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')


In [3]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'n_estimators': 6000,
    'num_leaves': 89,
    'max_depth': 11,
    'min_child_samples': 46,
    'reg_alpha': 9.4,
    'reg_lambda': 0.34,
    'min_split_gain': 1e-6,
    'subsample': 0.70,
    'subsample_freq': 3,
    'colsample_bytree': 0.62,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds_gbdt = [420, 666]
seeds_dart = [420]
n_splits = 5

EARLY_STOP = 150
MAX_SECONDS = 3300

t0 = time.time()

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

def cv_run(params, seeds, label):
    sum_oof = np.zeros(len(X), dtype=float)
    cnt_oof = np.zeros(len(X), dtype=float)
    sum_test = np.zeros(len(X_test), dtype=float)
    seeds_done = 0

    for s_i, seed in enumerate(seeds, start=1):
        p = {**params, 'random_state': seed}

        oof = np.full(len(X), np.nan, dtype=float)
        test_pred_sum = np.zeros(len(X_test), dtype=float)
        rmse_scores = []
        folds_done = 0

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
            if (time.time() - t0) > MAX_SECONDS:
                break

            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model = lgb.LGBMRegressor(**p)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(0)]
            )

            va_pred = model.predict(X_va)
            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f}')

            test_pred_sum += model.predict(X_test)
            folds_done += 1

        if folds_done == 0:
            break

        test_pred = test_pred_sum / folds_done

        filled = ~np.isnan(oof)
        oof_filled = np.clip(oof[filled], 0, 100)
        sum_oof[filled] += oof_filled
        cnt_oof[filled] += 1.0

        sum_test += np.clip(test_pred, 0, 100)
        seeds_done += 1

        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        if (time.time() - t0) > MAX_SECONDS:
            break

    denom = np.maximum(cnt_oof, 1.0)
    all_oof = np.clip(sum_oof / denom, 0, 100)

    if seeds_done > 0:
        all_test = np.clip(sum_test / seeds_done, 0, 100)
    else:
        all_test = np.zeros(len(X_test), dtype=float)

    filled_all = cnt_oof > 0
    if filled_all.any():
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all])))
    else:
        final_oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


gbdt_oof, gbdt_test, _ = cv_run(base_params, seeds_gbdt, 'GBDT')

dart_params = {
    **base_params,
    'boosting_type': 'dart',
    'drop_rate': 0.10,
    'skip_drop': 0.50,
    'max_drop': 50
}

dart_oof, dart_test, _ = cv_run(dart_params, seeds_dart, 'DART')

BLEND_W = 0.75
blend_oof = np.clip(BLEND_W * gbdt_oof + (1.0 - BLEND_W) * dart_oof, 0, 100)
blend_test = np.clip(BLEND_W * gbdt_test + (1.0 - BLEND_W) * dart_test, 0, 100)

blend_rmse = float(np.sqrt(mean_squared_error(y, blend_oof)))
print(f'BLEND OOF RMSE: {blend_rmse:.5f}')

y100 = (y >= 99.999).astype(int)
feat_cols = [c for c in ['study_hours', 'class_attendance', 'sleep_hours'] if c in X.columns]

X_clf_train = np.column_stack([blend_oof, blend_oof ** 2] + [X[c].to_numpy(dtype=float) for c in feat_cols])
X_clf_test = np.column_stack([blend_test, blend_test ** 2] + [X_test[c].to_numpy(dtype=float) for c in feat_cols])

final_test = blend_test

if int(y100.sum()) >= 20 and int(y100.sum()) <= (len(y100) - 20):
    clf = LogisticRegression(max_iter=500, class_weight='balanced')
    clf.fit(X_clf_train, y100)

    p100_train = clf.predict_proba(X_clf_train)[:, 1]
    p100_test = clf.predict_proba(X_clf_test)[:, 1]

    CEILING_STRENGTH = 0.15
    P100_THRESHOLD = 0.90
    NEAR_CEILING = 97.0

    adj_oof = blend_oof.copy()
    adj_test = blend_test.copy()

    m_tr = (blend_oof >= NEAR_CEILING) & (p100_train >= P100_THRESHOLD)
    m_te = (blend_test >= NEAR_CEILING) & (p100_test >= P100_THRESHOLD)

    adj_oof[m_tr] = blend_oof[m_tr] + CEILING_STRENGTH * p100_train[m_tr] * (100.0 - blend_oof[m_tr])
    adj_test[m_te] = blend_test[m_te] + CEILING_STRENGTH * p100_test[m_te] * (100.0 - blend_test[m_te])

    adj_oof = np.clip(adj_oof, 0, 100)
    adj_test = np.clip(adj_test, 0, 100)

    adj_rmse = float(np.sqrt(mean_squared_error(y, adj_oof)))
    print(f'CEILING OOF RMSE: {adj_rmse:.5f} | nudged train: {int(m_tr.sum())} test: {int(m_te.sum())}')

    final_test = adj_test
else:
    print('CEILING head skipped')

submission = pd.DataFrame({'id': test_ids, 'exam_score': np.clip(final_test, 0, 100)})
submission.to_csv('submission.csv', index=False)
print('Wrote submission.csv')


GBDT SEED 420 (1/2)
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2290]	valid_0's rmse: 8.73976
  Fold 1/5 RMSE: 8.73976
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2159]	valid_0's rmse: 8.74271
  Fold 2/5 RMSE: 8.74271
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2084]	valid_0's rmse: 8.73741
  Fold 3/5 RMSE: 8.73741
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2593]	valid_0's rmse: 8.75248
  Fold 4/5 RMSE: 8.75248
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2334]	valid_0's rmse: 8.77184
  Fold 5/5 RMSE: 8.77184
GBDT Seed 420 OOF RMSE: 8.74853 | Mean fold: 8.74884 (+/- 0.01259)
GBDT SEED 666 (2/2)
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[2401]	valid_0's rmse: 8.74023


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 1/5 RMSE: 8.77799


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 2/5 RMSE: 8.77513


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 3/5 RMSE: 8.77069
DART Seed 420 OOF RMSE: 8.77460 | Mean fold: 8.77460 (+/- 0.00300)
DART FINAL OOF RMSE: 8.77460
BLEND OOF RMSE: 13.46043
CEILING OOF RMSE: 13.46046 | nudged train: 1949 test: 1372
Wrote submission.csv
